In [3]:
import requests
import pandas as pd
from tqdm import tqdm



mappings = {
    '能源': [['国金证券'], ['石油行业','煤炭行业','采掘行业','燃气']],
    '有色金属': [['国金证券','华福证券'], ['有色金属','小金属','能源金属','贵金属']],
    '钢铁': [['华福证券'], ['钢铁行业','煤炭行业']],
    '化工': [['国金证券','国海证券'], ['化学制品','化纤行业','化学原料', '化肥行业', '塑料制品','橡胶制品', '非金属材料']],  
    
    '电力': [['国金证券'],['公用事业','电力行业','环保行业']], 
    '电力设备': [['国金证券','华福证券'], ['光伏设备','电池','风电设备','电网设备', '电源设备']], 
    
    '房地产': [['国金证券'],['房地产开发','房地产服务']], 
    '建材': [['国投证券','国信证券'], ['水泥建材','装修建材','玻璃玻纤']], 
    '建筑': [['国投证券'], ['工程建设','装修装饰']],  
    
    '金融': [['国金证券','华福证券'], ['银行', '证券','多元金融','保险']],
    '汽车': [['国金证券','华福证券'], ['汽车服务','汽车整车', '汽车零部件','交运设备']],
    '机械': [['国金证券'], ['专用设备', '通用设备','仪器仪表','工程机械']],
    '家电': [['国金证券','华福证券'], ['家电行业']],
    '电子': [['国金证券','华福证券'],['半导体', '消费电子','电子元件', '光学光电子','电子化学品']],
    
    '科技': [['国金证券','华福证券'], ['计算机设备',  '游戏', '通信设备', '通信服务', '互联网服务','软件开发','文化传媒']],
    '医药': [['国金证券','华福证券'], ['生物制品','医疗器械', '化学制药','医药商业','医疗服务','中药']],
    '食品饮料': [['国金证券','华福证券'], ['酿酒行业','食品饮料']],
    '农牧饲渔': [['国金证券'], ['农牧饲渔']],
    '轻工制造': [['华安证券','上海证券'], ['纺织服装', '家用轻工','造纸印刷', '包装材料']],
    '商贸零售': [['上海证券'], [ '商业百货', '美容护理', '珠宝首饰']],
    '社会服务': [['万联证券'], ['旅游酒店','教育','专业服务']], 
    '交通运输': [['国金证券','华福证券'], ['航运港口','航空机场', '物流行业','铁路公路']],
    
    '军工': [['中航证券'],['航天航空', '船舶制造']],
    
}






total_df = pd.DataFrame()
for page in tqdm(range(1,6)):
    url = 'https://reportapi.eastmoney.com/report/list'
    payload ={
        'industryCode': '*',
        'pageSize': '100',
        'industry': '*',
        'rating': '*',
        'ratingChange': '*',
        'beginTime': '2024-07-05',
        'endTime': '2024-07-10',
        'pageNo': str(page),
        'qType': '1'
    }

    r = requests.get(url, params = payload)
    df = pd.DataFrame(r.json()['data'])[['title','orgSName','publishDate','infoCode','industryCode','industryName','emRatingValue','researcher']]
    df['publishDate'] = pd.to_datetime(df['publishDate']).dt.date.astype(str)
    total_df = pd.concat([total_df,df])


total_df.loc[total_df['title'].str.contains('能源周'), 'industryName'] = '石油行业'
df = pd.DataFrame(mappings).T
def format_title(x):
    try:
        if len(x.split('：')) > 1:
            x = ':'.join(x.split('：')[1:])
        else:
            x = x.split('：')[-1]
    except:
        pass   
    return x
    
df['title'] = '' 
df['infoCode'] = ''


for i, row in df.iterrows():
    tmp = total_df[total_df['orgSName'].isin(row[0]) & total_df['industryName'].isin(row[1])]
    tmp['title'] = tmp['title'].apply(format_title)
    df.at[i, 'title'] = (tmp['publishDate'] + ' || ' + tmp['title']).to_list()
    df.at[i, 'infoCode'] = tmp['infoCode'].to_list()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:03<00:00,  1.46it/s]
C:\Users\Zhi\AppData\Local\Temp\ipykernel_13100\699978689.py:83: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tmp['title'] = tmp['title'].apply(format_title)


In [5]:
from requests_html import HTMLSession

def fetch_highlight(infocode_list):
    highlight_list = []
    for infocode in infocode_list:
        url = f'https://data.eastmoney.com/report/zw_industry.jshtml?infocode={infocode}'
        session = HTMLSession()  
        r = session.get(url)  
        highlight = r.html.find('div.ctx-content')[0].text.replace('\n','').split('风险提示')[0]
        highlight_list.append(highlight)
        # pdf_link = list(r.html.find('a.pdf-link')[0].links)[0]
    return highlight_list
#使用tqdm进度条
tqdm.pandas() 
df['highlight'] = df['infoCode'].progress_map(fetch_highlight)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [00:34<00:00,  1.49s/it]


In [38]:
import requests
import os


def make_query(highlight_list):
    num = len(highlight_list)
    query = f'你是一名资深的投资顾问, 请分析以下{num}篇券商行业研报的摘要内容，然后总结行业的发展现状, 并详细地列出推荐标的及理由：'
    for i in highlight_list:
        query = query + f'摘要{highlight_list.index(i)+1}：{i}'
    return query

def ask_qwen(query):
    api_key = 'sk-7397a9299d73435cb10290493bddee4e'
    url = 'https://dashscope.aliyuncs.com/api/v1/services/aigc/text-generation/generation'
    headers = {'Content-Type': 'application/json',
               'Authorization':f'Bearer {api_key}'}
    body = {
        'model': 'qwen-long',
        "input": {
            "messages": [
                {"role": "system","content": "You are an experienced investment manager."},
                {"role": "user","content": query}
            ]
        },
        "parameters": {"result_format": "message"}
    }
    response = requests.post(url, headers=headers, json=body)
    try:
        answer = response.json()['output']['choices'][0]['message']['content']
    except:
        answer = response.json()
    return answer
    

In [39]:
qwen_df = df[['title','highlight']].rename(columns = {'title':'标题','highlight':'摘要'})
qwen_df['千问读研报'] = qwen_df['摘要'].apply(make_query).progress_map(ask_qwen)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 23/23 [08:53<00:00, 23.18s/it]


In [41]:
from IPython.display import display, HTML

def to_str(x):
    try:
        x = '<br>'.join(x)
    except:
        pass
    return x



# qwen_df['标题'] = qwen_df['标题'].apply(to_str)
qwen_df['千问读研报'] = qwen_df['千问读研报'].str.replace('\n','')
display(HTML(qwen_df[['标题','千问读研报']].to_html(escape=False, justify='left')))

,标题,千问读研报
能源,2024-07-08 || 旺季来临，沙特原油供应却意外大幅下降,行业发展现状：根据摘要，能源市场目前呈现出以下特点：1. 原油价格：由于夏季需求旺季的到来和沙特供应的缩减，预期7、8月份油价将维持在高位。尽管本周油价略有波动，但OPEC+的出口量下降超出了市场的预期，这可能会支撑油价。2. 天然气价格：虽然美国天然气产量正在迅速恢复，但由于需求的增长，特别是预计夏季高温将增加用气需求，天然气价格有望在夏季反弹。然而，当前价格下跌可能预示着市场在调整供需平衡。3. 煤炭价格：煤炭价格略有下跌，国内产量稳定且进口有所增长。然而，由于充足的降水和水电对煤炭需求的抑制，近期煤炭需求并未显著增加。预测8月份随着降水正常化和高温天气，煤炭需求可能会有所增长。推荐标的：1. 原油相关公司：鉴于油价看涨，投资者可以关注上游石油勘探和生产公司，例如全球大型石油公司如沙特阿美、埃克森美孚等，这些公司的盈利能力有望受益于高油价。2. 天然气相关公司：考虑到夏季天然气需求增长的预期，投资于天然气生产和分销公司可能有潜力，如切萨皮克能源、荷兰皇家壳牌等。3. 煤炭相关公司：尽管近期需求受到压制，但预计8月份需求会增长，投资者可关注煤炭生产商，如中国神华、兖州煤业等，它们可能从未来需求增加中获益。理由：1. 高油价将直接提升石油公司的盈利，尤其对于成本控制良好的公司，利润空间较大。2. 天然气需求增长将推高价格，天然气生产商的销售收入有望增加。3. 随着天气变化和水电需求减少，煤炭作为补充能源的需求将上升，煤炭生产商的销售和股价可能随之改善。
有色金属,2024-07-08 || 锑减产加剧，重视稀土“类供改”催化下的底部布局机会2024-07-07 || 推荐关注Q3黄金股相对金价的估值修复机遇2024-07-07 || 美就业和经济数据疲软，降息预期升温，长期看好金铜铝锡锑,行业现状：1. 稀土行业：价格波动，受《稀土管理条例》影响，短期内可能出现抛售压力，但长期看，政策将规范市场，改善供给格局，提升稀土产业的话语权。资源端，北方稀土作为轻稀土龙头值得关注。2. 锑行业：锑锭价格稳定，库存处于历史低位，供应收缩预期增强，随着光伏玻璃旺季到来，价格有望上涨。湖南黄金作为弹性较大的标的被推荐。3. 钼行业：价格回调，但库存低，供需偏紧，未来有望因需求旺季和供应收缩而上涨。金钼股份被看好。4. 锡行业：价格上涨，冶炼厂减产，需求端有望受益于消费电子复苏，锡供需格局向好。锡业股份被推荐。5. 钨行业：价格下跌，市场交投承压，建议关注中钨高新。6. 其他小金属：五氧化二钒下跌，金属锗上涨。7. 铜行业：铜价上涨，库存累积，市场预期铜船货未到达美国，供应端出现变化。紫金矿业被推荐。8. 铝行业：铝价波动，供应端云南电解铝复产，需求端受淡季影响，但新能源汽车等产业需求稳健。中国铝业被看好。9. 金行业：金价上涨，受美国经济数据和降息预期影响。山东黄金、中金黄金、银泰黄金等被推荐。综合以上信息，推荐标的包括：- 北方稀土（稀土）- 湖南黄金（锑）- 金钼股份（钼）- 锡业股份（锡）- 中钨高新（钨）- 紫金矿业（铜、金、锂）- 中国铝业（铝）- 山东黄金、中金黄金、银泰黄金（金）这些标的因其在各自领域的领先地位、政策影响下的供需变化以及价格走势被看好。然而，投资决策应考虑整体市场环境、风险承受能力和投资策略，建议在充分研究和咨询专业投资顾问后进行决策。
钢铁,2024-07-08 || 铁水高位微降供需矛盾积压，政策托底预期渐强2024-07-08 || 南方高温持续，动力煤价格或完成筑底，短期高库存限制涨幅,摘要1主要讨论了钢铁行业的现状，铁矿石和双焦市场的情况，以及下游需求和政策影响。铁水产量在高位企稳，但淡季需求偏弱导致库存上升，压制钢价。国内货币政策保持灵活适度，而美国就业数据疲软增强了降息预期。推荐关注低估值、高股息、盈利稳定的普钢企业，如南钢股份、华菱钢铁、宝钢股份、新兴铸管，以及盈利弹性和高股息的特钢企业，如中信特钢、常宝股份、甬金股份。摘要2则主要分析了动力煤和炼焦煤市场，动力煤受到高温天气和电厂日耗回升的影响，煤价有望反弹，但高库存限制了涨幅。炼焦煤产量和库存有所波动，下游钢铁行业需求稳定但淡季消费减弱。推荐关注高长协业绩稳定的动力煤企业，如中国神华、陕西煤业、中煤能源，以及低估值、高股息的炼焦煤企业，如潞安环能、山西焦煤、冀中能源，并关注旺季可能量价齐升的公司，如山煤国际、晋控煤业。综合两篇摘要，钢铁行业面临供需矛盾，但政策支持和淡季过后的预期利好，动力煤和炼焦煤市场则受到天气、库存和下游需求的影响，价格有波动，但长期看供需关系可能收紧，价格有上升空间。因此，投资建议是：1. 钢铁行业：关注南钢股份、华菱钢铁、宝钢股份、新兴铸管、中信特钢、常宝股份、甬金股份，这些公司在低估值和稳定盈利方面具有优势。2. 动力煤行业：推荐中国神华、陕西煤业、中煤能源，这些公司受益于长协业绩稳定和估值修复。3. 炼焦煤行业：关注潞安环能、山西焦煤、冀中能源，它们在低估值和高股息方面有吸引力。4. 综合电力需求和旺季预期：留意兖矿能源、广汇能源，它们可能受益于电力需求增长。投资决策应结合市场动态、公司财务状况、行业趋势和风险偏好进行全面评估。
化工,2024-07-08 || 胎企开启新一轮出海，设备端有望充分受益2024-07-07 || 辛醇、MMA价格上涨，万华化学拟对IPDI装置技改扩能2024-07-07 || 中国5G用户普及率超60%，SpaceX将发起“北极星”计划首飞任务2024-07-07 || 万华化学POE一次性开车成功，龙头有望持续开疆拓土2024-07-07 || 龙头企业市占率提升，复合肥有望量利齐升2024-07-07 || 涤纶长丝价格上涨，看好行业供需向好及盈利修复,当前的轮胎行业在全球范围内呈现出稳定增长的趋势，尤其是欧美市场占据主导地位，而中国作为主要生产国，依赖出口且在全球市场份额不断提升。面对欧美地区的贸易政策不确定性，国内轮胎企业选择在海外建立生产基地以分散风险。同时，国产轮胎生产设备因其性价比高、服务好、响应速度快而受到国内轮胎制造商的青睐，行业内的设备制造商，如豪迈科技和软控股份，有望受益于这一趋势，实现快速发展。在化工行业，全球景气回升，中国化工企业表现出强劲的成本和效率优势，行业集中度提高，落后产能出清，新增产能逐渐达峰。受外需改善和内需稳定的影响，化工行业周期底部已过，尤其是那些具备成本优势、能够快速扩张产能的企业，如万华化学、轮胎制造商等，将迎来新的增长机遇。同时，新材料领域，如特种橡胶、高性能纤维等，也因下游需求增长和国产化率提升而备受关注。在投资建议上，推荐关注以下几个标的：1. 豪迈科技：作为全球轮胎模具和数控机床领域的领导者，公司将受益于轮胎企业扩产，有望实现稳定增长。2. 软控股份：作为全球领先的轮胎设备供应商，其整体解决方案能力将使其在轮胎企业扩产中获得较大收益。3. 万华化学：化工巨头，受益于行业景气度提升，其低成本扩张和多元化产品线将带动业绩增长。4. 轮胎制造商（玲珑轮胎、赛轮轮胎、森麒麟、通用股份、贵州轮胎）：这些公司因轮胎需求增长和出口市场扩展，预计将实现稳健增长。5. 其他化工企业，如卫星化学、远兴能源、宝丰能源、华鲁恒升、龙佰集团等，因成本优势和行业景气度提升，也值得关注。总体来说，轮胎和化工行业都处于增长阶段，尤其是一些具备规模优势、技术领先地位和全球化布局的企业，有望在行业发展中获得更大的市场份额和盈利能力。投资者应关注这些企业的业绩增长、产能扩张和技术创新等方面的发展动态。
电力,2024-07-08 || 2Q24水电电量高增，高库存压制煤价火电仍有看点,行业现状分析：1. 整体市场：本周股市波动，上证综指和创业板指均出现下跌，但公用事业板块表现相对强劲，尤其是水电板块，受益于预期的EPS增长和市场风险偏好降低。2. 煤炭与电力：煤炭板块小幅上涨，但火电发电量受到可再生能源增长的挤压，全国火电发电量同比下降。煤炭库存高企，对煤价形成压力，这可能有利于火电企业的盈利能力。3. 可再生能源：水电板块受厄尔尼诺现象影响，发电量大幅提升，显示出可再生能源的潜力和重要性。政策方面，国家对电化学储能和光伏产业的支持增强，推动了绿色能源的发展。4. 电力供应：闽粤联网工程提升了电力保供能力，新的国家标准和地方政策促进了电力互济和光伏应用，显示了电网互联和可再生能源在电力系统中的角色日益重要。推荐标的及理由：1. 浙能电力、皖能电力（火电板块）：这两家公司拥有发电资产位于电力供需紧张且市场竞争格局良好的地区，有望从煤价下跌和电力需求中受益。2. 长江电力（水电板块）：作为水电运营商的龙头，长江电力的股息率与国债收益率关联，随着国债收益率下降，其股息率有提升空间。同时，发电量增长也增强了其业绩预期。3. 中国核电（核电板块）：作为核电行业的领军企业，有望在国家政策支持下持续发展。4. 南网能源（新能源板块）：作为综合能源运营商，该公司在光伏、储能等领域具有广泛布局，将从新能源政策和市场需求增长中获益。以上投资建议基于当前市场情况和行业动态，投资者应考虑自身风险承受能力和市场变化，谨慎决策。
电力设备,2024-07-09 || 新兴市场需求专题（一）:光伏经济性凸显，新兴市场多点开花2024-07-08 || 逆变器超预期二连开启中报行情，电网设备再迎国际国内催化2024-07-08 || 产业周跟踪，法国海风百亿补贴落地，西门子推12亿欧元电网资本开支2024-07-08 || 稼动率回

In [37]:
qwen_df['千问读研报'][0]

C:\Users\Zhi\AppData\Local\Temp\ipykernel_13100\3918973568.py:1: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  qwen_df['千问读研报'][0]


{'code': 'AccessDenied',
 'message': 'Access denied.',
 'request_id': '8d7d6fb6-4206-96e3-a220-cadc1f99b5f9'}